# 🏛️ HLS Alpha Engine — Research Notebook

**ISCF** (Idiosyncratic Supply Chain Flow) & **MGD** (Real-Time Macro Growth Divergence)

## Table of Contents
1. [Setup](#1-setup)
2. [ISCF Signal Research](#2-iscf-signal-research)
3. [MGD Signal Research](#3-mgd-signal-research)
4. [Causal Validation](#4-causal-validation)
5. [Orthogonality Analysis](#5-orthogonality-analysis)
6. [Sharpe Waterfall](#6-sharpe-waterfall)


In [14]:
from __future__ import annotations
import math
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PLOTS = Path('plots')
PLOTS.mkdir(exist_ok=True)

from citadel_alpha import data_hls, signals_hls, causal, falsification as fal, constants as C
print('Modules loaded ✓')


Modules loaded ✓


In [15]:
# =============================================================================
# C++ Hot-Path Detection
# -----------------------------------------------------------------------------
# Attempt to import the compiled _citadel_alpha_cpp nanobind extension.
# If the extension is not built (e.g. constrained Windows environment without
# a compiler), Python-only fallback is selected automatically and a prominent
# WARNING is printed here and in every downstream computation cell.
# =============================================================================

import logging as _logging
import sys

_CPP_LOG_PREFIX = "[CPP-HOTPATH]"

try:
    import _citadel_alpha_cpp as _cpp_engine
    _CPP_AVAILABLE = True
    print(f"{_CPP_LOG_PREFIX} STATUS  : ENABLED")
    print(f"{_CPP_LOG_PREFIX} MODULE  : {_cpp_engine.__file__ if hasattr(_cpp_engine, '__file__') else '_citadel_alpha_cpp (built-in)'}")
    print(f"{_CPP_LOG_PREFIX} SIGNALS : compute_iscf, compute_mgd active")
except ImportError as _cpp_err:
    _CPP_AVAILABLE = False
    print(f"{_CPP_LOG_PREFIX} STATUS  : DISABLED — Python fallback will be used")
    print(f"{_CPP_LOG_PREFIX} REASON  : {_cpp_err}")
    print(f"{_CPP_LOG_PREFIX} FIX     : cmake -B build && cmake --build build && pip install -e .")
    print(f"{_CPP_LOG_PREFIX} NOTE    : All results remain numerically correct via pure-Python path.")

print()
print(f"{'C++ hot-path':>20} : {'✓ ACTIVE' if _CPP_AVAILABLE else '✗ UNAVAILABLE — Python fallback ACTIVE'}")


[CPP-HOTPATH] STATUS  : DISABLED — Python fallback will be used
[CPP-HOTPATH] REASON  : No module named '_citadel_alpha_cpp'
[CPP-HOTPATH] FIX     : cmake -B build && cmake --build build && pip install -e .
[CPP-HOTPATH] NOTE    : All results remain numerically correct via pure-Python path.

        C++ hot-path : ✗ UNAVAILABLE — Python fallback ACTIVE


In [16]:
# =============================================================================
# Hot-Path Dispatch Helpers
# -----------------------------------------------------------------------------
# _iscf_compute / _mgd_compute: dispatch to C++ or Python automatically.
# Gram-Schmidt residualization is always applied in Python (C++ does not
# include it — it handles only the cross-sectional core computation).
# Each call prints a one-time banner so the active path is always visible.
# =============================================================================

from citadel_alpha.signals_hls import gram_schmidt_residualise
from citadel_alpha.signals import SignalResult, _gaussian_rank_normalize as _grn

_HOTPATH_BANNER_SHOWN = {"iscf": False, "mgd": False}

def _iscf_compute(spot, deferred, rvol, next_ret, macro_beta, baseline):
    """Dispatch ISCF cross-sectional computation to C++ or Python fallback."""
    global _HOTPATH_BANNER_SHOWN
    if _CPP_AVAILABLE:
        if not _HOTPATH_BANNER_SHOWN["iscf"]:
            print(f"{_CPP_LOG_PREFIX} ISCF loop : using C++ hot-path + Python Gram-Schmidt")
            _HOTPATH_BANNER_SHOWN["iscf"] = True
        raw_d = _cpp_engine.compute_iscf(
            spot.astype(np.float64), deferred.astype(np.float64),
            rvol.astype(np.float64), macro_beta.astype(np.float64),
            next_ret.astype(np.float64),
        )
        raw = np.array(raw_d["raw_score"])
        # Python post-processing: Gram-Schmidt residualization against baselines
        raw_orth = gram_schmidt_residualise(raw, baseline)
        mu = float(np.mean(raw_orth)); sigma = float(np.std(raw_orth, ddof=1))
        z = (raw_orth - mu) / max(sigma, 1e-8)
        rank = _grn(z)
        ic_val = float(np.corrcoef(rank, next_ret.astype(np.float64))[0, 1]) if len(rank) > 1 else 0.0
        return SignalResult(signal_name="ISCF", raw_score=raw_orth, z_score=z,
                            rank_score=rank, ic=ic_val if np.isfinite(ic_val) else 0.0, icir=0.0)
    else:
        if not _HOTPATH_BANNER_SHOWN["iscf"]:
            print(f"{_CPP_LOG_PREFIX} ISCF loop : C++ unavailable — using Python fallback")
            _HOTPATH_BANNER_SHOWN["iscf"] = True
        return signals_hls.compute_iscf(spot, deferred, rvol, next_ret, macro_beta, baseline)


def _mgd_compute(pmi, cpi, emp, fwd_exp, roll_std, next_ret, baseline):
    """Dispatch MGD cross-sectional computation to C++ or Python fallback."""
    global _HOTPATH_BANNER_SHOWN
    if _CPP_AVAILABLE:
        if not _HOTPATH_BANNER_SHOWN["mgd"]:
            print(f"{_CPP_LOG_PREFIX} MGD  loop : using C++ hot-path + Python Gram-Schmidt")
            _HOTPATH_BANNER_SHOWN["mgd"] = True
        raw_d = _cpp_engine.compute_mgd(
            pmi.astype(np.float64), cpi.astype(np.float64),
            emp.astype(np.float64), fwd_exp.astype(np.float64),
            roll_std.astype(np.float64), next_ret.astype(np.float64),
            C.MGD_PMI_WEIGHT, C.MGD_INFLATION_WEIGHT, C.MGD_EMPLOYMENT_WEIGHT,
            2.0 / (C.MGD_SURPRISE_EMA_SPAN + 1.0),
        )
        raw = np.array(raw_d["raw_score"])
        raw_orth = gram_schmidt_residualise(raw, baseline)
        mu = float(np.mean(raw_orth)); sigma = float(np.std(raw_orth, ddof=1))
        z = (raw_orth - mu) / max(sigma, 1e-8)
        rank = _grn(z)
        ic_val = float(np.corrcoef(rank, next_ret.astype(np.float64))[0, 1]) if len(rank) > 1 else 0.0
        return SignalResult(signal_name="MGD", raw_score=raw_orth, z_score=z,
                            rank_score=rank, ic=ic_val if np.isfinite(ic_val) else 0.0, icir=0.0)
    else:
        if not _HOTPATH_BANNER_SHOWN["mgd"]:
            print(f"{_CPP_LOG_PREFIX} MGD  loop : C++ unavailable — using Python fallback")
            _HOTPATH_BANNER_SHOWN["mgd"] = True
        return signals_hls.compute_mgd(pmi, cpi, emp, fwd_exp, roll_std, next_ret, baseline)


## 1. Setup


In [17]:
N, T, SEED, WARMUP = 8, 2000, 42, 120
comm = data_hls.generate_commodity_panel(n=N, t=T, seed=SEED)
fx   = data_hls.generate_fx_panel(n=N, t=T, seed=SEED)
print(f'Commodity panel: T={T}, N={N}')
print(f'FX panel: T={T}, N={N}')


Commodity panel: T=2000, N=8
FX panel: T=2000, N=8


## 2. ISCF Signal Research


In [18]:
_HOTPATH_BANNER_SHOWN["iscf"] = False  # reset so banner fires once per run
iscf_ics, iscf_pnl = [], []
for i in range(WARMUP, T):
    bl = np.column_stack([comm.trend_returns[i], comm.momentum_returns[i], comm.carry_returns[i]])
    res = _iscf_compute(comm.spot[i], comm.deferred[i], comm.rvol[i],
                        comm.forward_returns[i], comm.macro_beta[i], bl)
    iscf_ics.append(res.ic)
    iscf_pnl.append(float(np.mean(res.rank_score * comm.forward_returns[i])))

iscf_ic = np.array(iscf_ics)
iscf_pnl = np.array(iscf_pnl)
iscf_sr = float(np.mean(iscf_pnl)/np.std(iscf_pnl,ddof=1)*math.sqrt(252))
print(f'ISCF Mean IC: {np.mean(iscf_ic):.4f} | SR: {iscf_sr:.3f}')


[CPP-HOTPATH] ISCF loop : C++ unavailable — using Python fallback
ISCF Mean IC: 0.0893 | SR: 3.771


In [19]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14,7), sharex=True)
win=60; ric=np.convolve(iscf_ic, np.ones(win)/win, mode='same')
ax1.plot(iscf_ic, alpha=0.2, color='#2196F3')
ax1.plot(ric, color='#2196F3', lw=1.8, label=f'ISCF 60d MA IC')
ax1.axhline(C.IC_FLOOR, color='green', ls=':', lw=0.9, label=f'IC floor={C.IC_FLOOR}')
ax1.axhline(0, color='black', lw=0.5)
ax1.set_ylabel('IC'); ax1.legend(); ax1.grid(alpha=0.3)
ax1.set_title('ISCF — Rolling IC (Metals/Energy Futures)', fontweight='bold')
cum = np.cumsum(iscf_pnl); rm = np.maximum.accumulate(cum); dd = cum - rm
ax2.plot(cum, color='#2196F3', lw=1.5)
ax2.fill_between(range(len(dd)), dd, 0, color='red', alpha=0.35, label='Drawdown')
ax2.set_ylabel('Cum PnL'); ax2.set_xlabel('Day'); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS/'iscf_ic_pnl.png', dpi=150, bbox_inches='tight')
plt.close(); print('Saved plots/iscf_ic_pnl.png ✓')


Saved plots/iscf_ic_pnl.png ✓


## 3. MGD Signal Research


In [20]:
_HOTPATH_BANNER_SHOWN["mgd"] = False  # reset so banner fires once per run
mgd_ics, mgd_pnl = [], []
for i in range(WARMUP, T):
    bl = np.column_stack([fx.trend_returns[i], fx.momentum_returns[i], fx.carry_returns[i]])
    res = _mgd_compute(fx.pmi_surprise[i], fx.cpi_surprise[i], fx.emp_surprise[i],
                       fx.fwd_expectation[i], fx.roll_std[i], fx.forward_returns[i], bl)
    mgd_ics.append(res.ic)
    mgd_pnl.append(float(np.mean(res.rank_score * fx.forward_returns[i])))

mgd_ic = np.array(mgd_ics)
mgd_pnl = np.array(mgd_pnl)
mgd_sr = float(np.mean(mgd_pnl)/np.std(mgd_pnl,ddof=1)*math.sqrt(252))
print(f'MGD Mean IC: {np.mean(mgd_ic):.4f} | SR: {mgd_sr:.3f}')


[CPP-HOTPATH] MGD  loop : C++ unavailable — using Python fallback
MGD Mean IC: 0.0896 | SR: 3.942


In [21]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14,7), sharex=True)
rmc=np.convolve(mgd_ic, np.ones(60)/60, mode='same')
ax1.plot(mgd_ic, alpha=0.2, color='#FF5722')
ax1.plot(rmc, color='#FF5722', lw=1.8, label='MGD 60d MA IC')
ax1.axhline(C.IC_FLOOR, color='green', ls=':', lw=0.9)
ax1.axhline(0, color='black', lw=0.5)
ax1.set_ylabel('IC'); ax1.legend(); ax1.grid(alpha=0.3)
ax1.set_title('MGD — Rolling IC (FX Forward Panel)', fontweight='bold')
cum2=np.cumsum(mgd_pnl); rm2=np.maximum.accumulate(cum2); dd2=cum2-rm2
ax2.plot(cum2, color='#FF5722', lw=1.5)
ax2.fill_between(range(len(dd2)), dd2, 0, color='red', alpha=0.35, label='Drawdown')
ax2.set_ylabel('Cum PnL'); ax2.set_xlabel('Day'); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS/'mgd_ic_pnl.png', dpi=150, bbox_inches='tight')
plt.close(); print('Saved plots/mgd_ic_pnl.png ✓')


Saved plots/mgd_ic_pnl.png ✓


## 4. Causal Validation


In [22]:
n_c = min(500, len(iscf_pnl))
# Use PnL time-series as signal proxy (avoids near-zero mean(rank_score) issue)
# iscf_pnl / mgd_pnl carry the causal structure embedded by data_hls v2.1
iscf_ret_cs = np.array([float(np.mean(comm.forward_returns[i]))
                          for i in range(WARMUP, WARMUP + n_c)])
mgd_ret_cs  = np.array([float(np.mean(fx.forward_returns[i]))
                          for i in range(WARMUP, WARMUP + n_c)])
conf = np.column_stack([
    comm.session_dummies[WARMUP:WARMUP + n_c],
    comm.vix_proxy[WARMUP:WARMUP + n_c].reshape(-1, 1),
])
iscf_cs = causal.run_causal_stack(
    'ISCF', iscf_pnl[:n_c], iscf_ret_cs, confounders=conf, n_bootstrap=100)
mgd_cs  = causal.run_causal_stack(
    'MGD',  mgd_pnl[:n_c],  mgd_ret_cs,  n_bootstrap=100)
print(iscf_cs.summary)
print()
print(mgd_cs.summary)


Signal: ISCF
  Step 1 — Granger VARX : F=1.601 p=0.2028 lag=2 → ✗ FAIL
  Step 2 — CMI          : stat=0.0000 retained=0.00% → ✗ FAIL
  Step 3 — DoWhy Placebo: p=0.0000 → ✗ FAIL
  Step 3 — Policy Inv.  : p=0.0000 → ✗ FAIL
  γ (Causal Confidence) : 0.00
  Recommendation        : ✗ REJECT

Signal: MGD
  Step 1 — Granger VARX : F=0.716 p=0.4890 lag=2 → ✗ FAIL
  Step 2 — CMI          : stat=0.0000 retained=0.00% → ✗ FAIL
  Step 3 — DoWhy Placebo: p=0.0000 → ✗ FAIL
  Step 3 — Policy Inv.  : p=0.0000 → ✗ FAIL
  γ (Causal Confidence) : 0.00
  Recommendation        : ✗ REJECT


In [23]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, cs, color in zip(axes, [iscf_cs, mgd_cs], ['#2196F3','#FF5722']):
    steps = ['Granger\nVARX','CMI\nTest','Placebo\nTest','Policy\nInvariance']
    # Use 1-p so bars above α=0.05 line = PASS for all four tests
    vals = [
        1.0 - cs.granger.p_value,
        cs.cmi.alpha_retained_fraction,
        1.0 - cs.dowhy.placebo_p_value,
        cs.dowhy.policy_invariance_p_value,
    ]
    passes = [cs.granger.passes, cs.cmi.passes,
               cs.dowhy.placebo_passes, cs.dowhy.policy_passes]
    clrs = ['#4CAF50' if p else '#F44336' for p in passes]
    ax.bar(steps, vals, color=clrs, edgecolor='white')
    ax.axhline(0.05, color='orange', ls='--', lw=1.2, label='α=0.05')
    ax.set_ylim(0, 1.1); ax.set_ylabel('Test Statistic / p-value')
    ax.set_title(f'{cs.signal_name} Causal Stack (γ={cs.final_gamma:.2f})', fontweight='bold')
    ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS/'causal_validation.png', dpi=150, bbox_inches='tight')
plt.close(); print('Saved plots/causal_validation.png ✓')


Saved plots/causal_validation.png ✓


## 5. Orthogonality Analysis


In [24]:
labels = ['ISCF','MGD','Trend','Momentum','Carry']
# Use strategy P&L series (not per-period IC scalars which are noise-dominated)
min_T = min(len(iscf_pnl), len(mgd_pnl), T - WARMUP)
matrix_data = np.column_stack([
    iscf_pnl[-min_T:],
    mgd_pnl[-min_T:],
    comm.trend_returns[WARMUP:WARMUP + min_T, 0],
    comm.momentum_returns[WARMUP:WARMUP + min_T, 0],
    comm.carry_returns[WARMUP:WARMUP + min_T, 0],
])
r2 = np.corrcoef(matrix_data.T) ** 2
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(r2, cmap='RdYlGn_r', vmin=0, vmax=0.3)
plt.colorbar(im, ax=ax, label='R²')
ax.set_xticks(range(5)); ax.set_yticks(range(5))
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticklabels(labels)
ax.axhline(1.5, color='black', lw=1.5, ls='--', alpha=0.5)
ax.axvline(1.5, color='black', lw=1.5, ls='--', alpha=0.5)
for i in range(5):
    for j in range(5):
        ax.text(j, i, f'{r2[i,j]:.2f}', ha='center', va='center', fontsize=9,
                color='white' if r2[i, j] > 0.2 else 'black')
off_diag_max = r2[np.ix_([0, 1], [2, 3, 4])].max()
ax.set_title(
    f'ISCF & MGD Orthogonality R² (max={off_diag_max:.3f}, thresh<{C.MAX_R2_ORTHOGONALITY})',
    fontweight='bold')
plt.tight_layout()
plt.savefig(PLOTS/'orthogonality.png', dpi=150, bbox_inches='tight')
plt.close(); print('Saved plots/orthogonality.png ✓')


Saved plots/orthogonality.png ✓


## 6. Sharpe Waterfall


In [25]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, sr, name, color in zip(axes, [iscf_sr, mgd_sr], ['ISCF','MGD'], ['#2196F3','#FF5722']):
    wf = fal.sharpe_waterfall(sr)
    stages = ['Gross SR', 'After TC', 'After Overfit', 'After Slippage', 'Net SR']
    vals = [wf.gross_sr, wf.sr_after_tc, wf.sr_after_overfit, wf.sr_after_slippage, wf.net_sr]
    clrs = [color if v>=C.WALKFORWARD_SHARPE_TARGET else '#9E9E9E' for v in vals]
    bars = ax.bar(stages, vals, color=clrs, edgecolor='white')
    ax.axhline(C.WALKFORWARD_SHARPE_TARGET, color='orange', ls='--', lw=1.5, label=f'Walk-fwd floor={C.WALKFORWARD_SHARPE_TARGET}')
    for b, v in zip(bars, vals):
        ax.text(b.get_x()+b.get_width()/2, v+0.02, f'{v:.2f}', ha='center', fontweight='bold')
    ax.set_title(f'{name} Sharpe Waterfall (t-stat={wf.t_stat:.1f})', fontweight='bold')
    ax.legend(); ax.grid(axis='y', alpha=0.3); ax.set_ylim(0, max(vals)*1.3)
plt.tight_layout()
plt.savefig(PLOTS/'sharpe_waterfall.png', dpi=150, bbox_inches='tight')
plt.close(); print('Saved plots/sharpe_waterfall.png ✓')


Saved plots/sharpe_waterfall.png ✓


In [26]:
# Signal health reports
for name, ic_a, sr in [('ISCF',iscf_ic,iscf_sr),('MGD',mgd_ic,mgd_sr)]:
    h = fal.signal_health_report(name, ic_a, sr)
    print(h.summary); print()


Signal: ISCF
  Mean IC       : 0.0893  (✓ floor=0.02)
  ICIR          : 0.2383  (✗ floor=0.5)
  Half-life     : 31.8d (✓ range=[21,63]d)
  Gross SR      : 3.771  (✓ floor=2.0)
  t-stat        : 59.86  (✓ floor=3.0)
  Net SR (est.) : 2.921
  RETIRE?       : ✓ NO

Signal: MGD
  Mean IC       : 0.0896  (✓ floor=0.02)
  ICIR          : 0.2370  (✗ floor=0.5)
  Half-life     : 33.7d (✓ range=[21,63]d)
  Gross SR      : 3.942  (✓ floor=2.0)
  t-stat        : 62.58  (✓ floor=3.0)
  Net SR (est.) : 3.092
  RETIRE?       : ✓ NO

